In [1]:
import json
import random
from pathlib import Path
from collections import defaultdict

In [2]:
ROOT = Path.cwd()
QUESTION_PATH = ROOT / "all_questions.jsonl"
ROOT

WindowsPath('e:/2026/字节和我的心脏只有一个可以跳动/rag_core')

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

e:\miniconda\envs\langchain2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def load_jsonl(path) -> list[dict]:
    rows  = []
    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows
    
questions = load_jsonl(QUESTION_PATH)
print(type(questions))
print(questions[0])
print(type(questions[0]))
print(len(questions))
print(questions[0].keys())

<class 'list'>
{'question_id': 'cpp_c++_syntax_project_deep_dive_1c18e81950', 'role': 'cpp', 'role_label': 'C++工程师', 'topic': 'C++ Syntax', 'topic_id': 'cpp:cxx_syntax', 'question_type': 'project_deep_dive', 'difficulty': 'hard', 'question': '请描述一个你在实际项目中使用 C++ 模板元编程（TMP）解决复杂问题的经历。具体说明你为何选择模板元编程而非运行时多态，并阐述方案权衡、遇到的编译错误或性能问题，以及最终如何优化。', 'expected_answer': '项目背景：简述问题场景，如需要在编译期进行类型计算或策略选择。；方案选择：对比模板元编程与运行时多态（如虚函数）的优缺点，强调编译期计算的优势（如零运行时开销）。；权衡与排障：讨论编译错误（如模板递归深度限制）或性能瓶颈（如代码膨胀），并说明解决方法。；优化与复盘：如何通过 SFINAE、C++17 的 if constexpr 或概念（concepts）简化模板代码，并评估实际性能提升。', 'reference_points': ['项目背景：简述问题场景，如需要在编译期进行类型计算或策略选择。', '方案选择：对比模板元编程与运行时多态（如虚函数）的优缺点，强调编译期计算的优势（如零运行时开销）。', '权衡与排障：讨论编译错误（如模板递归深度限制）或性能瓶颈（如代码膨胀），并说明解决方法。', '优化与复盘：如何通过 SFINAE、C++17 的 if constexpr 或概念（concepts）简化模板代码，并评估实际性能提升。'], 'follow_up_angles': ['如果项目规模扩大，如何管理模板代码的可读性和维护性？', '在现代 C++（C++20）中，你会如何用概念（concepts）重构此模板代码？', '请分享一个因模板误用导致的隐蔽运行时错误案例及调试过程。'], 'common_mistakes': ['只描述项目背景，没有说清自己的职责和贡献。', '没有量化效果，也没有说明方案取舍。', '遇到问题时只给结果，不讲排查路径和

In [9]:
from langchain_core.documents import Document
def json_to_question_document(obj: dict, source_file: str) -> Document:
    """
    把 _questions.jsonl 的一行转换成题目 Document。
    用于 interview_questions collection。
    """
    question_id = obj.get("question_id", "")
    role = obj.get("role", "")
    topic = obj.get("topic", "")
    topic_id = obj.get("topic_id", "")
    question_type = obj.get("question_type", "")
    difficulty = obj.get("difficulty", "")
    question = obj.get("question", "")
    expected_answer = obj.get("expected_answer", "")
    reference_points = obj.get("reference_points", [])
    follow_up_angles = obj.get("follow_up_angles", [])
    common_mistakes = obj.get("common_mistakes", [])
    tags = obj.get("tags", [])
    assesses_topic_ids = obj.get("assesses_topic_ids", [])

    page_content = (
        f"[文档类型]: interview_question\n"
        f"[题目ID]: {question_id}\n"
        f"[岗位]: {role}\n"
        f"[知识点]: {topic}\n"
        f"[主题ID]: {topic_id}\n"
        f"[题型]: {question_type}\n"
        f"[难度]: {difficulty}\n"
        f"[题目]: {question}\n"
        f"[参考答案]: {expected_answer}\n"
        # f"[评分要点]: {_join_list(reference_points)}\n"
        # f"[追问方向]: {_join_list(follow_up_angles)}\n"
        # f"[常见错误]: {_join_list(common_mistakes)}\n"
        # f"[考察主题]: {_join_list(assesses_topic_ids)}\n"
        # f"[标签]: {_join_list(tags)}\n"
    )

    metadata = {
        "doc_type": "interview_question",
        "source": source_file,
        "question_id": question_id,
        "role": role,
        "topic": topic,
        "topic_id": topic_id,
        "question_type": question_type,
        "difficulty": difficulty,
        "source_file": obj.get("source_file", source_file),
        "source_line": int(obj.get("source_line", 0) or 0),
    }

    return Document(page_content=page_content, metadata=metadata)

In [10]:
def load_documents_and_ids_from_jsonl(file_path:Path) -> list[dict]:
    """
    打开文件地址，读取文件内容，并把每一行的 JSON 字符串转换成 Document 对象，最后返回一个 Document 对象的列表。
    """
    documents = []
    ids = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            doc = json_to_question_document(obj,source_file=file_path.name)
            documents.append(doc)
            doc_id = f"{file_path.stem}_{len(documents)}"
            ids.append(doc_id)
            
    return documents,ids

In [11]:
docs, ids = load_documents_and_ids_from_jsonl(QUESTION_PATH)
print(docs[0])
print(type(docs))
print(ids[0])

page_content='[文档类型]: interview_question
[题目ID]: cpp_c++_syntax_project_deep_dive_1c18e81950
[岗位]: cpp
[知识点]: C++ Syntax
[主题ID]: cpp:cxx_syntax
[题型]: project_deep_dive
[难度]: hard
[题目]: 请描述一个你在实际项目中使用 C++ 模板元编程（TMP）解决复杂问题的经历。具体说明你为何选择模板元编程而非运行时多态，并阐述方案权衡、遇到的编译错误或性能问题，以及最终如何优化。
[参考答案]: 项目背景：简述问题场景，如需要在编译期进行类型计算或策略选择。；方案选择：对比模板元编程与运行时多态（如虚函数）的优缺点，强调编译期计算的优势（如零运行时开销）。；权衡与排障：讨论编译错误（如模板递归深度限制）或性能瓶颈（如代码膨胀），并说明解决方法。；优化与复盘：如何通过 SFINAE、C++17 的 if constexpr 或概念（concepts）简化模板代码，并评估实际性能提升。
' metadata={'doc_type': 'interview_question', 'source': 'all_questions.jsonl', 'question_id': 'cpp_c++_syntax_project_deep_dive_1c18e81950', 'role': 'cpp', 'topic': 'C++ Syntax', 'topic_id': 'cpp:cxx_syntax', 'question_type': 'project_deep_dive', 'difficulty': 'hard', 'source_file': 'cpp_project.jsonl', 'source_line': 1}
<class 'list'>
all_questions_1


In [15]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
def init_chroma_db(persist_dir, collection_name):
    emb = HuggingFaceEmbeddings(
        model_name = "sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs = {"device":"cuda"},
        encode_kwargs={"normalize_embeddings": True}
    )

    db = Chroma(
        persist_directory=persist_dir,
        collection_name=collection_name,
        embedding_function=emb,
    )  
    return db

def add_documents_to_chroma(db, documents, ids):
    db.add_documents(documents, ids=ids)
    return db

In [17]:
persist_dir = "chroma_all"
collection_name = "question"
db = init_chroma_db(
    persist_dir=persist_dir,
    collection_name=collection_name,
)
# db = add_documents_to_chroma(
#     db=db,
#     documents=docs,
#     ids=ids,
# )
batch_size = 1000
for i in range(0,len(docs),batch_size):
    batch_docs = docs[i:i+batch_size]
    batch_ids = ids[i:i+batch_size]

    db.add_documents(batch_docs, ids=batch_ids)
print(db._collection.count())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7921.91it/s]


8350


In [19]:
result = db._collection.get(
    ids = ["all_questions_1"]
)
print(type(result))
for doc_id , doc_text , mete in zip(result["ids"] , result["documents"],result["metadatas"]):
    print(f"ID: {doc_id}\npage_content:\n{doc_text}\nmetedata: {mete}\n")

<class 'dict'>
ID: all_questions_1
page_content:
[文档类型]: interview_question
[题目ID]: cpp_c++_syntax_project_deep_dive_1c18e81950
[岗位]: cpp
[知识点]: C++ Syntax
[主题ID]: cpp:cxx_syntax
[题型]: project_deep_dive
[难度]: hard
[题目]: 请描述一个你在实际项目中使用 C++ 模板元编程（TMP）解决复杂问题的经历。具体说明你为何选择模板元编程而非运行时多态，并阐述方案权衡、遇到的编译错误或性能问题，以及最终如何优化。
[参考答案]: 项目背景：简述问题场景，如需要在编译期进行类型计算或策略选择。；方案选择：对比模板元编程与运行时多态（如虚函数）的优缺点，强调编译期计算的优势（如零运行时开销）。；权衡与排障：讨论编译错误（如模板递归深度限制）或性能瓶颈（如代码膨胀），并说明解决方法。；优化与复盘：如何通过 SFINAE、C++17 的 if constexpr 或概念（concepts）简化模板代码，并评估实际性能提升。

metedata: {'source_file': 'cpp_project.jsonl', 'topic': 'C++ Syntax', 'role': 'cpp', 'source': 'all_questions.jsonl', 'difficulty': 'hard', 'source_line': 1, 'question_type': 'project_deep_dive', 'question_id': 'cpp_c++_syntax_project_deep_dive_1c18e81950', 'topic_id': 'cpp:cxx_syntax', 'doc_type': 'interview_question'}



In [ ]:
from dataclasses import dataclass
@dataclass
class Question:
    question_id: str
    topic: str
    difficulty: str = "medium"
    question: str
    

In [ ]:
""" 
完整的题单的数据结构 plan - plan中的一道题目 - 题目
"""
class InterviewPlanItem:
    index: int
    question_id: str
    topic: str
    question: Question

class InterviewPlan:
    plan_id: str
    topics: list[str]
    num_questions: int
    items: list[InterviewPlanItem]

In [ ]:
""" 
单题评分结构体
主问题和追问都可以使用
"""
class EvaluationResult:
    score: int
    reason: str
    
    hit_points: list[str] = []
    missing_points: list[str] = []
    mistakes: list[str] = []
    suggestion: str | None = None

In [ ]:
""" 
一道题的完整记录
"""
class InterviewTurn:
    index: int
    question_id: str
    topic: str

    question: str
    answer: str
    followup_question: str | None = None
    followup_answer: str | None = None

    evaluation: EvaluationResult | None = None
    status: str = "not_started"

In [ ]:
""" 
用来记录完整面试流程
plan 记录了完整的面试题单 
turns 记录了每道题目的完整记录/问答情况 包括评分结果
session.plan.items[session.current_index] 就是当前题目
"""
class InterviewSession:
    session_id: str
    user_id: str
    status: str # in_progress / completed
    plan = InterviewPlan

    current_index: int
    turns: list[InterviewTurn] = []
    created_time: str
    updated_time: str
    finished_time: str
    final_report: dict

In [ ]:
import uuid
import datetime
def init_interview_state(user_id:str, interview_plan:InterviewPlan) -> InterviewSession:
    session = InterviewSession()
    turns = []
    for item in interview_plan.items:
        turn = InterviewTurn(
            index=item.index,
            question_id=item.question_id,
            topic=item.topic,
            question=item.question.question,
            status="not_started"
        )
        turns.append(turn)
    # uuid是python标准库 用来生成通用唯一标识符
    # pythonuuid.uuid4() 会随机生成一个唯一id
    session.session_id = str(uuid.uuid4()),
    session.user_id = user_id
    session.status = "in_progress"
    session.plan = interview_plan
    session.current_index = 0 
    session.turns = turns
    session.created_time = datetime.now().isoformat(),
    session.updated_time = datetime.now().isoformat(),


In [ ]:
def build_interview_plan(db, topics: list[str], num_quesitions, difficulty: str="medium") -> InterviewPlan:
    random_seed = 1
    rng = random.Random(random_seed)
    all_candidates: list[Question] = []
    for topic in topics:
        topic_question = 